<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%209/Question_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates**

# **Task 1: Structural Probability and Properties**
The Beta distribution acts as a flexible prior over the domain $[0, 1]$, where the shape parameters $\alpha$ and $\beta$ can be conceptualized as pseudo-counts of successes (clicks) and failures (non-clicks).Run the following code block in your Google Colab notebook to visualize the probability density function (PDF) under different parameter states.

In [1]:
# @title Execute this cell to generate the Beta PDF visualization
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define the evaluation grid for theta
theta_vals = np.linspace(0, 1, 500)

# Scenarios to plot
scenarios = [
    {"alpha": 1, "beta": 1, "name": "Uninformative: Beta(1,1)", "color": "gray", "dash": "dash"},
    {"alpha": 2, "beta": 8, "name": "Right-skewed: Beta(2,8)", "color": "blue", "dash": "solid"},
    {"alpha": 8, "beta": 2, "name": "Left-skewed: Beta(8,2)", "color": "red", "dash": "solid"}
]

fig = go.Figure()

for s in scenarios:
    # Compute the probability density function pointwise
    y_vals = beta.pdf(theta_vals, s["alpha"], s["beta"])
    fig.add_trace(go.Scatter(
        x=theta_vals, y=y_vals,
        mode='lines',
        name=s["name"],
        line=dict(color=s["color"], dash=s["dash"], width=2.5)
    ))

fig.update_layout(
    title="Probability Density Function (PDF) of the Beta Distribution",
    xaxis_title="Click-Through Rate (θ)",
    yaxis_title="Probability Density f(θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)

fig.show()


### Interpretation of Center of Mass
The expected value (center of mass) of a $\text{Beta}(\alpha, \beta)$ distribution is governed by the ratio $\frac{\alpha}{\alpha + \beta}$.

*   **Uninformative state ($\alpha=1, \beta=1$)**: The mass is distributed uniformly across the entire domain. The derivative of the density with respect to $\theta$ is zero everywhere, expressing absolute uncertainty.
*   **Right-skewed state ($\alpha=2, \beta=8$)**: Because $\beta > \alpha$, the denominator dominates, shifting the center of mass toward $0$ (specifically, $\mathbb{E}[\Theta] = \frac{2}{2+8} = 0.20$). This encodes a prior belief that the CTR is low.
*   **Left-skewed state ($\alpha=8, \beta=2$)**: Because $\alpha > \beta$, the numerator drives the mass toward $1$ (specifically, $\mathbb{E}[\Theta] = \frac{8}{8+2} = 0.80$), encoding a strong prior belief that the advertisement converts highly.

# **Task 2: Sequential Likelihood and Joint History**

1. Isolated Likelihood ContributionFor a single standalone observation $y_k \in \{0, 1\}$ at timeline step $k$, given the conditional click probability $\theta$, the likelihood function is derived from the Bernoulli probability mass function (PMF):$$L(y_k \mid \theta) = P(Y_k = y_k \mid \Theta = \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$


2. Joint Likelihood FunctionLet $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the history vector of observations. Assuming that each user interaction is conditionally independent given the true hidden parameter $\theta$ (the local independence assumption), the joint likelihood is the product of the individual likelihood components:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} \theta^{y_i} (1 - \theta)^{1 - y_i}$$Using the laws of exponents to simplify:$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{\left(\sum_{i=1}^{k} y_i\right)} (1 - \theta)^{\left(\sum_{i=1}^{k} (1 - y_i)\right)}$$Let $S_k = \sum_{i=1}^{k} y_i$ represent the total number of clicks observed up to step $k$, and let $F_k = \sum_{i=1}^{k} (1 - y_i) = k - S_k$ represent the total number of non-clicks. The joint likelihood simplifies to:$$L(\mathbf{y}^{(k)} \mid \theta) = \theta^{S_k} (1 - \theta)^{F_k}$$

# **Task 3: Closed-Form Analytical Updates (Conjugacy)**

1. Analytical Proof of Beta-Binomial ConjugacyWe wish to derive the exact analytical form of the posterior distribution at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, given that the prior state at step $k-1$ is distributed as $\text{Beta}(\alpha_{k-1}, \beta_{k-1})$.Proof:By Bayes' Theorem, the posterior density is proportional to the product of the newest isolated likelihood contribution and the running prior density:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{0}^{1} L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \, d\theta}$$Let us substitute the explicit functional forms of the likelihood (from Task 2) and the prior density:$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$$$f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1}$$Substitute these terms directly into the numerator of our expression:$$\text{Numerator} = \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \cdot \left[ \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$Group common base terms together by algebraically adding their exponents:$$\text{Numerator} = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \cdot \theta^{(\alpha_{k-1} - 1 + y_k)} \cdot (1 - \theta)^{(\beta_{k-1} - 1 + 1 - y_k)}$$Simplify the exponents:$$\text{Numerator} = \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \cdot \theta^{(\alpha_{k-1} + y_k) - 1} \cdot (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$Now, substitute this simplified numerator expression back into the complete fraction:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{\frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}}{\int_{0}^{1} \frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1} \, d\theta}$$The constant scalar $\frac{1}{\mathrm{B}(\alpha_{k-1}, \beta_{k-1})}$ can be factored completely out of the integral in the denominator. This allows it to cancel out perfectly with the identical term in the numerator:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{\theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}}{\int_{0}^{1} \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1} \, d\theta}$$By definition, the standard Beta function $\mathrm{B}(u, v)$ is defined via the integral:$$\mathrm{B}(u, v) = \int_{0}^{1} t^{u-1} (1-t)^{v-1} \, dt$$Looking at the denominator integral, we can assign matching dummy parameters:$$u = \alpha_{k-1} + y_k$$$$v = \beta_{k-1} + 1 - y_k$$Thus, the definite integral evaluates exactly to $\mathrm{B}(\alpha_{k-1} + y_k, \, \beta_{k-1} + 1 - y_k)$. Replacing the denominator with this value gives:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{1}{\mathrm{B}(\alpha_{k-1} + y_k, \, \beta_{k-1} + 1 - y_k)} \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$This matches the exact definition of a Beta distribution probability density function. Hence, the posterior distribution remains closed within the Beta family.The arithmetic recursive parameters are:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k) \quad \bl$$2. Derivation of the Posterior MeanWe wish to derive the expected value $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}]$ directly from our updated density function.Proof:By the definition of the mathematical expectation for a continuous random variable:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$Substitute the verified normalized distribution expression derived above:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \int_{0}^{1} \theta \cdot \left[ \frac{1}{\mathrm{B}(\alpha_k, \beta_k)} \theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1} \right] \, d\theta$$Factor the normalizing constant outside the integral and combine the $\theta$ terms:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{1}{\mathrm{B}(\alpha_k, \beta_k)} \int_{0}^{1} \theta^{(\alpha_k + 1) - 1} (1 - \theta)^{\beta_k - 1} \, d\theta$$Using the definition of the Beta integral again, the integral term evaluates to $\mathrm{B}(\alpha_k + 1, \beta_k)$:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\mathrm{B}(\alpha_k + 1, \beta_k)}{\mathrm{B}(\alpha_k, \beta_k)}$$The relationship between the Beta function and the Gamma function $\Gamma(z)$ is given by:$$\mathrm{B}(u, v) = \frac{\Gamma(u)\Gamma(v)}{\Gamma(u+v)}$$Expanding both the numerator and denominator using this Gamma relationship yields:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\left[ \frac{\Gamma(\alpha_k + 1)\Gamma(\beta_k)}{\Gamma(\alpha_k + 1 + \beta_k)} \right]}{\left[ \frac{\Gamma(\alpha_k)\Gamma(\beta_k)}{\Gamma(\alpha_k + \beta_k)} \right]}$$Cancel out the shared term $\Gamma(\beta_k)$ from both components:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\Gamma(\alpha_k + 1)}{\Gamma(\alpha_k)} \cdot \frac{\Gamma(\alpha_k + \beta_k)}{\Gamma(\alpha_k + \beta_k + 1)}$$Using the fundamental recursive property of the Gamma function, $\Gamma(z+1) = z\Gamma(z)$, we can simplify the fractions:$$\frac{\Gamma(\alpha_k + 1)}{\Gamma(\alpha_k)} = \frac{\alpha_k\Gamma(\alpha_k)}{\Gamma(\alpha_k)} = \alpha_k$$$$\frac{\Gamma(\alpha_k + \beta_k)}{\Gamma(\alpha_k + \beta_k + 1)} = \frac{\Gamma(\alpha_k + \beta_k)}{(\alpha_k + \beta_k)\Gamma(\alpha_k + \beta_k)} = \frac{1}{\alpha_k + \beta_k}$$Multiplying these two simplified expressions gives the final equation for the Posterior Mean:$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)} = \mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k} \quad \re  $$

# **Task 4: Dynamic Shifting Mechanics**



### The occurrence of an event shifts the peak (mode) of the posterior density distribution through exact arithmetic incrementation:

*   When a click occurs ($y_k = 1$): The shape parameter updates to $\alpha_k = \alpha_{k-1} + 1$, while $\beta_k = \beta_{k-1}$. The density function is multiplied by the linear likelihood function $L(y_k=1 \mid \theta) = \theta$, which suppresses the density near $0$ and increases it near $1$. This shifts the peak of the distribution to the right.
*   When a non-click occurs ($y_k = 0$): The shape parameter updates to $\alpha_k = \alpha_{k-1}$, while $\beta_k = \beta_{k-1} + 1$. The density function is multiplied by $L(y_k=0 \mid \theta) = (1 - \theta)$, suppressing the density near $1$ and shifting the peak of the distribution to the left.

### Contrast Against Non-Conjugate Setups
In non-conjugate setups (such as the 2PL IRT model), multiplying the prior by the likelihood function yields a posterior distribution that does not match any standard, named probability family. Because there is no clean algebraic form, we cannot determine its properties analytically. Instead, the platform must use computationally intensive numerical integration (like a Riemann grid or Gauss-Hermite quadrature) at every single step to re-normalize the distribution and approximate its statistics.

Under this conjugate Beta-Binomial framework, we completely bypass numerical integration. The parameters update via simple addition, allowing the platform to track updates instantly with minimal computational overhead.

# **Task 5: Running Point Estimators**

At any step $k$, we can extract exact point estimates directly using closed-form algebraic expressions:1. Running Posterior Mean (Expected A Posteriori - EAP)The posterior mean represents the minimum mean squared error (MMSE) estimator:$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$2. Running Maximum A Posteriori (MAP)The MAP estimate represents the mode (the most likely value) of the distribution. For a Beta distribution where $\alpha_k > 1$ and $\beta_k > 1$, this peak is located exactly at:$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$

# **Task 6: Performance Tracking and Convergence Analysis**
The following script simulates $n=100$ impressions for a true underlying click-through rate $\theta_{\text{true}} = 0.35$, starting from an uninformative uniform prior ($\alpha_0=1, \beta_0=1$).

In [2]:
import numpy as np
import plotly.graph_objects as go

# Set seed for reproducible simulation results
np.random.seed(42)

# --- Simulation Setup ---
theta_true = 0.35
n_impressions = 100

# Base parameters for the initial uniform prior Beta(1,1)
alpha_param = 1.0
beta_param = 1.0

# Arrays to store the timeline progression
history_bayes_mean = [alpha_param / (alpha_param + beta_param)]
# For Beta(1,1), the MAP is technically undefined/uniform; we initialize it at the mode center 0.5
history_map_est = [0.5]
steps = list(range(n_impressions + 1))

# --- Sequential Update Simulation Loop ---
for k in range(1, n_impressions + 1):
    # 1. Simulate a single Bernoulli trial interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0

    # 2. Perform the analytical closed-form update steps
    alpha_param += y_k
    beta_param += (1 - y_k)

    # 3. Calculate point estimators derived in Task 5
    bayes_mean = alpha_param / (alpha_param + beta_param)

    if alpha_param > 1 and beta_param > 1:
        map_est = (alpha_param - 1) / (alpha_param + beta_param - 2)
    else:
        map_est = 0.5

    # Append the calculated values to tracking arrays
    history_bayes_mean.append(bayes_mean)
    history_map_est.append(map_est)

# --- Interactive Line Chart Generation ---
fig = go.Figure()

# Add Tracking Line for Posterior Mean
fig.add_trace(go.Scatter(
    x=steps, y=history_bayes_mean,
    mode='lines+markers',
    name='Posterior Mean (EAP)',
    line=dict(color='darkblue', width=2),
    marker=dict(size=4)
))

# Add Tracking Line for MAP
fig.add_trace(go.Scatter(
    x=steps, y=history_map_est,
    mode='lines+markers',
    name='Maximum A Posteriori (MAP)',
    line=dict(color='darkorange', width=2, dash='dash'),
    marker=dict(size=4)
))

# Add Static Reference Line for True CTR Value
fig.add_trace(go.Scatter(
    x=steps, y=[theta_true] * len(steps),
    mode='lines',
    name='True CTR (θ = 0.35)',
    line=dict(color='crimson', width=1.5, dash='dot')
))

fig.update_layout(
    title="Sequential Tracking of Advertisement CTR via Beta-Binomial Updates",
    xaxis_title="Timeline Step (Impression k)",
    yaxis_title="Estimated Click-Through Rate (θ)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# **Convergence Analysis**
As the number of observed impressions $k$ approaches $100$, the distance between both estimators and the true parameter $\theta_{\text{true}} = 0.35$ decreases significantly. Early in the simulation, you will notice wider fluctuations because each early observation represents a large percentage of the total data collected.As data accumulates, the likelihood function narrows and dominates the update process. The initial prior parameters $(\alpha_0=1, \beta_0=1)$ quickly lose their influence relative to the overwhelming amount of real-world evidence. The variance of the posterior distribution continuously shrinks, demonstrating that the platform's confidence in its measurement increases with every impression.